In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1499, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 1499 (delta 93), reused 0 (delta 0), pack-reused 1359 (from 1)
Receiving objects: 100% (1499/1499), 626.68 MiB | 53.39 MiB/s, done.
Resolving deltas: 100% (921/921), done.
Updating files: 100% (222/222), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.0 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorizationImpressions_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorizationImpressions_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'twra_study.db'
json_name = 'twra_best_params.json'

config = {
    'tune_parameters': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name': json_name,
    'copy_prev_best_params': False,
    'save_github': True,
    'candidates': '20',
    'n_trials': 100
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.GraphBased.TwoWayRankAggregationRecommender import TwoWayRankAggregationRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'twra_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_twra(optuna_trial):
    recommender_instance = TwoWayRankAggregationRecommender(URM_train)
    
    f_params = {
        'topK': optuna_trial.suggest_int("f_topK", 20, 80),  
        'alpha': optuna_trial.suggest_float("f_alpha", 0.8, 1.8),
        'beta': optuna_trial.suggest_float("f_beta", 0.1, 0.7),
        'normalize_similarity': True 
    }
    
    b_params = {
        'topK': optuna_trial.suggest_int("b_topK", 10, 1000),
        'alpha': optuna_trial.suggest_float("b_alpha", 0.0, 2.0),
        'beta': optuna_trial.suggest_float("b_beta", 0.0, 2.0),
        'normalize_similarity': True
    }
    convex_lambda = optuna_trial.suggest_float("convex_lambda", 0.0, 1.0)
    
    recommender_instance.fit(
        base_f_algorithm='rp3beta',
        base_b_algorithm='rp3beta',
        convex_lambda=convex_lambda,
        f_params=f_params,
        b_params=b_params
    )
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time

if config['tune_parameters']:
    study = optuna.create_study(direction='maximize', study_name='twra_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_twra, n_trials=config['n_trials'])

[I 2025-12-29 21:42:51,770] Using an existing study with name 'twra_Optimization' instead of creating a new one.


Previous trials found: 66
RP3betaRecommender: Similarity column 6969 (100.0%), 1718.82 column/sec. Elapsed time 4.05 sec
RP3betaRecommender: Similarity column 27095 (100.0%), 472.60 column/sec. Elapsed time 57.33 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 3.56 min. Users per second: 127


[I 2025-12-29 21:47:37,828] Trial 66 finished with value: 0.24299508944775847 and parameters: {'f_topK': 24, 'f_alpha': 1.7999377426028063, 'f_beta': 0.4533064341934297, 'b_topK': 405, 'b_alpha': 1.706482011039328, 'b_beta': 0.7052361131042493, 'convex_lambda': 0.3380587126307868}. Best is trial 52 with value: 0.2660150979495087.


RP3betaRecommender: Similarity column 6969 (100.0%), 1876.62 column/sec. Elapsed time 3.71 sec
RP3betaRecommender: Similarity column 27095 (100.0%), 516.53 column/sec. Elapsed time 52.46 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 3.22 min. Users per second: 140


[I 2025-12-29 21:51:56,213] Trial 67 finished with value: 0.26146451998415887 and parameters: {'f_topK': 26, 'f_alpha': 1.6730754219691057, 'f_beta': 0.5111703748404053, 'b_topK': 302, 'b_alpha': 0.43233158529035726, 'b_beta': 0.6927675282318237, 'convex_lambda': 0.358650803389028}. Best is trial 52 with value: 0.2660150979495087.


...

RP3betaRecommender: Similarity column 6969 (100.0%), 1816.35 column/sec. Elapsed time 3.84 sec
RP3betaRecommender: Similarity column 27095 (100.0%), 401.53 column/sec. Elapsed time 1.12 min
EvaluatorHoldout: Processed 27067 (100.0%) in 5.14 min. Users per second: 88


[I 2025-12-30 05:21:13,516] Trial 164 finished with value: 0.27068529128912944 and parameters: {'f_topK': 23, 'f_alpha': 1.134245088492506, 'f_beta': 0.11068298740567484, 'b_topK': 722, 'b_alpha': 0.43066626742258163, 'b_beta': 0.7404410119364748, 'convex_lambda': 0.29284083601415156}. Best is trial 106 with value: 0.27281173827547656.


RP3betaRecommender: Similarity column 6969 (100.0%), 1757.46 column/sec. Elapsed time 3.97 sec
RP3betaRecommender: Similarity column 27095 (100.0%), 397.34 column/sec. Elapsed time 1.14 min
EvaluatorHoldout: Processed 27067 (100.0%) in 5.16 min. Users per second: 87


[I 2025-12-30 05:27:55,621] Trial 165 finished with value: 0.26881240800560163 and parameters: {'f_topK': 23, 'f_alpha': 1.1300339769133747, 'f_beta': 0.10628964231090844, 'b_topK': 727, 'b_alpha': 0.4488554386166612, 'b_beta': 0.7400239524373474, 'convex_lambda': 0.2634809678935156}. Best is trial 106 with value: 0.27281173827547656.


CPU times: user 6h 55min 22s, sys: 53min 27s, total: 7h 48min 49s
Wall time: 7h 45min 5s


In [17]:
if config['save_github'] and config['tune_parameters']:
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )

    if config['save_json']:
        j_name = config['json_name']
        l_path = f"/kaggle/working/{j_name}"
        r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
        
        best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
        
        try:
            for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
                if s.best_trial and s.best_trial.value > best_val:
                    best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
        except:
            best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

        formatted_params = {
            "base_f_algorithm": "rp3beta",
            "base_b_algorithm": "rp3beta",
            "convex_lambda": best_params.get("convex_lambda")
        }

        for key, value in best_params.items():
            if key.startswith("f_"):
                new_key = key.replace("f_", "f_rp3_")
                formatted_params[new_key] = value
            elif key.startswith("b_"):
                new_key = key.replace("b_", "b_rp3_")
                formatted_params[new_key] = value
    
        prev_val, exists = -float('inf'), False
        try:
            prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
            exists = True
        except:
            pass
    
        if not exists or best_val > prev_val:
            print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
            with open(l_path, 'w') as f:
                json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": formatted_params}, f, indent=4)
            if config.get('save_github'):
                upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
        else:
            print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

Updating JSON. Best: 0.272812 (Prev: 0.266015)
